# 🚀 CALYPSO-RAG: Production QLoRA Fine-Tuning Pipeline for GATE CS
### Domain-Adapted Mathematical Reasoning Engine on `Qwen2.5-1.5B-Instruct`

This notebook trains a **4-Bit QLoRA (Quantized Low-Rank Adaptation)** reasoning adapter on authentic **GATE Computer Science & IT (1990 - 2026)** problems.

#### ⚙️ Technical Architecture:
- **Base Model**: `Qwen/Qwen2.5-1.5B-Instruct`
- **Quantization**: 4-bit NormalFloat (NF4) with Double Quantization (`bitsandbytes`)
- **PEFT/LoRA**: Rank $r=16$, Alpha $\alpha=32$, Dropout $0.05$
- **Target Modules**: `q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj`
- **Prompt Template**: ChatML `<|im_start|>system...<|im_start|>user...<|im_start|>assistant...`

## 1. Install Dependencies

In [ ]:
!pip install -q -U torch transformers datasets peft bitsandbytes trl accelerate

## 2. Verify GPU Acceleration

In [ ]:
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## 3. Clone Repository & Extract Real Training Dataset

In [ ]:
!git clone https://github.com/piyush23-eng/CALYPSO-RAG.git
%cd CALYPSO-RAG
!python scripts/prepare_training_data.py

## 4. Run QLoRA Fine-Tuning

In [ ]:
!python scripts/train_qlora.py \
    --base_model Qwen/Qwen2.5-1.5B-Instruct \
    --data_path data/train_gate_cs_dataset.jsonl \
    --output_dir models/calypso_gate_qlora \
    --epochs 4 \
    --batch_size 2 \
    --grad_accum 4 \
    --lr 2e-4

## 5. Test the Trained Model on Unseen Problem

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from peft import PeftModel
import torch

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_PATH = "models/calypso_gate_qlora/final_adapter"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

# Merge LoRA Adapter
trained_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
trained_model.eval()

pipe = pipeline("text-generation", model=trained_model, tokenizer=tokenizer)

prompt = (
    "<|im_start|>system\nYou are an expert GATE Computer Science reasoning model.<|im_end|>\n"
    "<|im_start|>user\n"
    "Consider a hard disk with a rotational speed of 15000 rpm. Adjacent track seek time is 1 ms. "
    "Initially on track 0. 400 sectors per track. Transfer 10 randomly located sectors in tracks 5, 12, and 7. "
    "What is the total data transfer time in milliseconds?<|im_end|>\n"
    "<|im_start|>assistant\n"
)

response = pipe(prompt, max_new_tokens=400, temperature=0.2, top_p=0.9)
print(response[0]["generated_text"].split("<|im_start|>assistant\n")[-1])